How much of the scatter that we observe in the difference between the expected $V(0.4R_{26})$ (from the MaNGA rotation curve fits of Ravi+26) and what we calculate from our DESI observations is due to the known uncertainties in the DESI fiber positioning?

In [59]:
import numpy as np

from astropy.table import Table
from astropy.coordinates import SkyCoord, Distance
from astropy.cosmology import FlatLambdaCDM
from astropy.wcs import WCS
import astropy.constants as const
import astropy.units as u

import os
# rotcurvepath = os.path.join(os.environ['HOME'], 'desi/RotationCurves/spirals')
# rotcurvepath = os.path.join(os.environ['HOME'], 'RotationCurves/spirals')
rotcurvepath = os.path.join(os.environ['HOME'], 'Documents/Research/Rotation_curves/RotationCurves/spirals')
import sys
sys.path.insert(1, rotcurvepath)
from dark_matter_mass_v1 import rot_fit_BB

from tqdm import tqdm

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.rc('font', size=12)

In [58]:
h = 1
H0 = 100*h*u.km/u.s/u.Mpc

c = const.c.to('km/s')

q0 = 0.2

MANGA_SPAXEL_SIZE = 0.5*u.arcsec

# Redrock systematic duplicate redshift uncertainty (from Lan+23)
dv_sys = 7 # km/s
dz_sys = dv_sys/c.value
#### ultimately dz = (1+z) dv/c, but we will multiply this by (1+z) for each observation

cosmo = FlatLambdaCDM(H0=H0, Om0=0.3151)

# Import data

## Original Iron healpix query results
We need to know the location of each fiber.

In [4]:
# tiron = Table.read('/global/cfs/projectdirs/desi/science/td/pv/tfgalaxies/desi_pv_tf_iron_healpix.fits')
tiron = Table.read('/Users/kdouglass/Documents/Research/data/DESI/Y1/desi_pv_tf_iron_healpix.fits')
tiron[:5]

TARGETID,TARGET_RA,TARGET_DEC,HEALPIX,SURVEY,Z,ZERR,ZWARN,DELTACHI2,FILENAME,PVTYPE,SGA_ID,RA,DEC
int64,float64,float64,int64,bytes4,float64,float64,int64,float64,bytes65,bytes3,int64,float64,float64
2852147603439621,198.369130660983,36.5372037049171,10475,main,0.815976335547845,7.38513168100107e-05,4,0.128754377365112,iron/healpix/main/dark/104/10475/redrock-main-dark-10475.fits,EXT,649377,198.36913066098333,36.537203704917076
2399148812795907,198.371733180003,36.4994335406917,10475,main,1.11088784970434,7.48767797671894e-05,4,7.9473560154438,iron/healpix/main/bright/104/10475/redrock-main-bright-10475.fits,EXT,649377,198.37173318000336,36.499433540691676
2399382443917318,184.845242475328,49.8157304793777,10995,main,1.14739342108157,0.000146302276719084,4,2.56771463155746,iron/healpix/main/bright/109/10995/redrock-main-bright-10995.fits,EXT,1008911,184.84524247532795,49.81573047937771
2399634072797192,184.341289722203,70.8283725474297,11965,main,1.51703376230705,6.28979649962091e-05,4,4.76254060305655,iron/healpix/main/bright/119/11965/redrock-main-bright-11965.fits,EXT,241234,184.34128972220284,70.82837254742968
2852141710442505,123.256011148025,36.2652948002806,6448,main,0.00787379494184006,3.4714052819995e-05,0,22.1719104201402,iron/healpix/main/dark/64/6448/redrock-main-dark-6448.fits,EXT,31591,123.25601114802525,36.26529480028061


### Update all Redrock uncertainties to account for 7 km/s statistical uncertainty

In [5]:
tiron['ZERR_MOD'] = np.sqrt(tiron['ZERR']**2 + (dz_sys*(1 + tiron['Z']))**2)

## Iron VI file

In [45]:
iron_VI = Table.read('../../../TF/Y1/iron_VI.txt', format='ascii.commented_header')

### Remove targets that don't pass VI

In [46]:
VI_pass = np.ones(len(tiron), dtype=bool)

for targetid in iron_VI['TARGETID']:

    if targetid in tiron['TARGETID']:
        VI_pass = VI_pass & (tiron['TARGETID'] != targetid)

tiron_goodVI = tiron[VI_pass]

## SGA

This is the catalog that we used for the TF analysis.

In [6]:
# using tf v15 catalog for update rot vel uncertainties
# tf_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/tfgalaxies/Y1/DESI-DR1_TF_pv_cat_v15.fits')
tf_targets = Table.read('/Users/kdouglass/Documents/Research/data/DESI/Y1/DESI-DR1_TF_pv_cat_v15.fits')
tf_targets[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT,Z_DESI,ZERR_DESI,V_0p4R26,V_0p4R26_ERR,PHOTSYS,MORPHTYPE_AI,JOHN_VI,Z_DESI_CMB,G_MAG_SB26_CORR,R_MAG_SB26_CORR,Z_MAG_SB26_CORR,G_MAG_SB26_ERR_CORR,R_MAG_SB26_ERR_CORR,Z_MAG_SB26_ERR_CORR,MU_ZCMB,R_ABSMAG_SB26,MU_ZCMB_ERR,R_ABSMAG_SB26_ERR,DWARF,GOOD_MORPH,GOOD_VEL,R_ABSMAG_SB26_TF,R_ABSMAG_SB26_TF_ERR,R_ABSMAG_SB26_TF_ERR_STAT,R_ABSMAG_SB26_TF_ERR_SYS,MU_TF,MU_TF_ERR,LOGDIST,LOGDIST_ERR,MAIN
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,mag,,mag,,,,,,,,,,,mag,mag,
float64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,float64,float64,float64,float64,bytes1,bytes10,bytes6,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,bool,bool,float64,float64,float64,float64,float64,float64,float64,float64,bool
38.0,SGA-2020 38,SDSSJ140638.88+393138.1,3558705,211.66194,39.5272341,S?,81.63,0.35318315,0.37757218,0.08108376,24.72948,18.361,False,LEDA-20181114,12,SDSSJ140638.88+393138.1,1,True,211.66194,39.5272341,0.35318315,2115p395,211.66203166014844,39.52724032794356,0.41757938,SB26,82.21093,0.41431504,211.6619547015994,39.52729608884245,8.520181,5.252184,4.0750155,3.3951538,3.3006833,4.4037066,5.5453897,6.693271,7.8023033,8.999919,10.213078,11.41192,12.527382,19.246052,18.406073,17.931826,18.841032,18.092857,17.659027,18.566164,17.886087,17.47688,18.383362,17.749825,17.355968,18.261652,17.655558,17.284552,18.168955,17.588947,17.231817,18.101948,17.547142,17.20673,18.055267,17.516266,17.18719,18.023865,1

In [7]:
SGA_dict = {}

for i in range(len(tf_targets)):
    
    SGA_dict[int(tf_targets['SGA_ID'][i])] = i

## SDSS MaNGA best fits

Nitya's fitting from 2025 on DR17 (Ravi+26)

In [9]:
MaNGA_fits = Table.read('../Elliptical_sphdisk_refitspirals_BPT_illustris_v11.fits', # Ravi et al. (2026)
                        format='fits')
MaNGA_fits[:5]

plate,ifudsgn,plateifu,mangaid,versdrp2,versdrp3,verscore,versutil,versprim,platetyp,srvymode,objra,objdec,ifuglon,ifuglat,ifura,ifudec,ebvgal,nexp,exptime,drp3qual,bluesn2,redsn2,harname,frlplug,cartid,designid,cenra,cendec,airmsmin,airmsmed,airmsmax,seemin,seemed,seemax,transmin,transmed,transmax,mjdmin,mjdmed,mjdmax,gfwhm,rfwhm,ifwhm,zfwhm,mngtarg1,mngtarg2,mngtarg3,catidnum,plttarg,manga_tileid,nsa_iauname,ifudesignsize,ifutargetsize,ifudesignwrongsize,z,zmin,zmax,szmin,szmax,ezmin,ezmax,probs,pweight,psweight,psrweight,sweight,srweight,eweight,esweight,esrweight,nsa_field,nsa_run,nsa_camcol,nsa_version,nsa_nsaid,nsa_nsaid_v1b,nsa_z,nsa_zdist,nsa_sersic_absmag,nsa_elpetro_absmag,nsa_elpetro_amivar,nsa_sersic_mass,nsa_elpetro_mass,nsa_elpetro_ba,nsa_elpetro_phi,nsa_extinction,nsa_elpetro_th50_r,nsa_petro_th50,nsa_petro_flux,nsa_petro_flux_ivar,nsa_elpetro_flux,nsa_elpetro_flux_ivar,nsa_sersic_ba,nsa_sersic_n,nsa_sersic_phi,nsa_sersic_th50,nsa_sersic_flux,nsa_sersic_flux_ivar,smoothness_score,nsa_elpetro_th90,v_sys,v_sys_err,ba,ba_err,x0,x0_err,y0,y0_err,phi,phi_err,v_max,v_max_err,r_turn,r_turn_err,chi2,alpha,alpha_err,Rmax,M,M_err,fit_flag,Sigma_disk,Sigma_disk_err,R_disk,R_disk_err,rho_bulge,rho_bulge_err,R_bulge,R_bulge_err,M90_disk,M90_disk_err,M_disk,M_disk_err,chi2_disk,WF50,WF50_err,DL_ttype,vis_tidal,b,b_err,M_R90,M_R90_err,fit_function,A_g,A_r,logH2,R90_kpc,v_3p5,v_3p5_err,NSA_plate,NSA_fiberID,NSA_MJD,logH2_CG,logH2_CG_err,logH2_M,Z,Z_err,M_Z,M_Z_err,grad_Z,grad_Z_err,Z_0,Z_0_err,SFR,sSFR,SFR_err,sSFR_err,Flux_OII_3726,Flux_OII_3726_Err,Flux_OII_3728,Flux_OII_3728_Err,Flux_OIII_4958,Flux_OIII_4958_Err,Flux_OIII_5006,Flux_OIII_5006_Err,Flux_NII_6547,Flux_NII_6547_Err,Flux_NII_6583,Flux_NII_6583_Err,Flux_Ha_6562,Flux_Ha_6562_Err,Flux_Hb_4861,Flux_Hb_4861_Err,CMD_class,rabsmag_NSA,param_H2,param_H2_err,Z_map,Z_err_map,M_Z_map,M_Z_err_map,logHI_R90,Mvir,Mvir_err,star_sigma,star_sigma_err,dipole_moment,Rgal,vflag_VF,nsa_elpetro_log_mass,rabsmag,param_H2_R90,logHe,sphd_rho_c,sphd_rho_c_err,sphd_R_scale,sphd_R_scale_err,sphd_Sigma_d,sphd_Sigma_d_err,sphd_R_d,sphd_R_d_err,sphd_M_star,sphd_M_star_err,sphd_chi2,mhq_R_scale,mhq_R_scale_err,mhq_M_star,mhq_M_star_err,mhq_gamma,mhq_gamma_err,mhq_chi2,hq_R_scale,hq_R_scale_err,hq_M_star,hq_M_star_err,hq_chi2,sph_rho_c,sph_rho_c_err,sph_R_scale,sph_R_scale_err,sph_M_star,sph_M_star_err,sph_chi2,sum_M_star,sum_M_star_err,BPT_class,spiral_mask,elliptical_mask,Age_LW_Re_fit,Age_MW_Re_fit,ZH_LW_Re_fit,ZH_MW_Re_fit,M90_disk_h,M90_disk_err_h,sphd_M_star_err_h,sphd_M_star_h,V_R90,V_R90_err,e_M_star_R90,e_M_star_R90_err,e_M_star_R90_h,e_M_star_R90_err_h,v_eff,v_eff_err,logHI,logHI_err,HI_catalog,logHI_h,logHI_err_h,N2Ha,O3Hb,nsa_u_r,illustris_match,illustris_dist,illustris_u_r,illustris_Mstar,illustris_Mstar_Sal,illustris_Vmax,illustris_flag,illustris_MR90,MH2_S14_vol,MHI_S14_vol,MHtot
int64,bytes32,bytes32,bytes32,bytes32,bytes32,bytes32,bytes32,bytes32,bytes32,bytes32,float64,float64,float64,float64,float64,float64,float64,int64,float64,int64,float64,float64,bytes53,int64,bytes32,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,int64,float64,float64,float64,float64,int64,int64,int64,int64,bytes32,int64,bytes19,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,int64,bytes6,int64,int64,float64,float64,float64[7],float64[7],float64[7],float64,float64,float64,float64,float64[7],float64,float64,float64[7],float64[7],float64[7],float64[7],float64,float64,float64,float64,float64[7],float64[7],float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,

## SDSS MaNGA cross-match file

In [8]:
SGA_MaNGA = Table.read('../../MaNGA_SGA_crossmatch_2022-06-28.txt', 
                       format='ascii.commented_header')

SGA_MaNGA[:5]

plate,ifudsgn,plateifu,SGA_ID
int64,int64,str11,int64
8716,3703,8716-3703,927743
9086,6104,9086-6104,1003700
11967,1902,11967-1902,196166
8613,6104,8613-6104,376271
10001,3703,10001-3703,937846


### Add TF catalog info to SGA-MaNGA table

In [10]:
SGA_MaNGA['Z_DESI'] = np.nan
SGA_MaNGA['ZERR_DESI'] = np.nan
SGA_MaNGA['R26'] = np.nan
SGA_MaNGA['BA'] = np.nan
SGA_MaNGA['PA'] = np.nan
SGA_MaNGA['V_0p4R26'] = np.nan
SGA_MaNGA['V_0p4R26_ERR'] = np.nan

for i in range(len(SGA_MaNGA)):
    
    # Find the row in SGA for this galaxy
    try:
        SGA_idx = SGA_dict[SGA_MaNGA['SGA_ID'][i]]
        
        # Set the redshift of this galaxy
        SGA_MaNGA['Z_DESI'][i] = tf_targets['Z_DESI'][SGA_idx]
        SGA_MaNGA['ZERR_DESI'][i] = tf_targets['ZERR_DESI'][SGA_idx]
        
        # # Transfer R26 over to the SGA_MaNGA table
        SGA_MaNGA['R26'][i] = 0.5*tf_targets['D26'][SGA_idx]
        
        # # Transfer b/a over to the SGA_MaNGA table
        SGA_MaNGA['BA'][i] = tf_targets['BA'][SGA_idx]
        
        # # Transfer phi over to the SGA_MaNGA table
        SGA_MaNGA['PA'][i] = tf_targets['PA'][SGA_idx]

        # Transfer over velocity and err at 0.4 R26
        SGA_MaNGA['V_0p4R26'][i] = tf_targets['V_0p4R26'][SGA_idx]
        SGA_MaNGA['V_0p4R26_ERR'][i] = tf_targets['V_0p4R26_ERR'][SGA_idx]

    except:
        continue

In [15]:
print(len(SGA_MaNGA), len(np.unique(SGA_MaNGA['SGA_ID'])))

8422 8285


# Filter the table to remove objects without a rotational velocity

In [18]:
hasV = np.isfinite(SGA_MaNGA['V_0p4R26'])

SGA_MaNGA_dr1 = SGA_MaNGA[hasV]

print(len(SGA_MaNGA_dr1), len(np.unique(SGA_MaNGA_dr1['SGA_ID'])))

276 270


# Add the rotational velocities and distances to the SDSS MaNGA - SGA cross-match table

In [19]:
# add sky fiber dist in radians
SGA_MaNGA_dr1['SKY_FIBER_DIST'] = (0.4*SGA_MaNGA_dr1['R26']*u.arcmin).to('radian')

In [20]:
SGA_MaNGA_dr1.show_in_notebook()

         interactive tables it is recommended to use dedicated tools like:
         - https://github.com/bloomberg/ipydatagrid
         - https://docs.bokeh.org/en/latest/docs/user_guide/interaction/widgets.html#datatable
         - https://dash.plotly.com/datatable [warnings]


idx,plate,ifudsgn,plateifu,SGA_ID,Z_DESI,ZERR_DESI,R26,BA,PA,V_0p4R26,V_0p4R26_ERR,SKY_FIBER_DIST
,,,,,,,,,,,,rad
0,9045,3702,9045-3702,29152,0.033320236763565,2.479367968152881e-05,0.31793898344039917,0.6588062047958374,168.80343627929688,136.65487767299953,11.458479382493653,3.699388054319129e-05
1,8615,1901,8615-1901,1265162,0.0197616666129208,2.4068447363858412e-05,0.2447134554386139,0.9234866499900818,75.16405487060547,142.94633456308551,29.494479148008217,2.8473703475574915e-05
2,7958,3702,7958-3702,1279893,0.0370122558474312,2.820956877908207e-05,0.4380972981452942,0.6277525424957275,142.59364318847656,219.66869829372382,14.440870762250226,5.0974935311510876e-05
3,11834,6104,11834-6104,1002687,0.0440140222262614,2.6774924836553197e-05,0.2553322911262512,0.4506562352180481,161.87460327148438,152.18986377456818,6.239301721442626,2.9709261112091895e-05
4,12081,12702,12081-12702,929612,0.0418290423138136,2.9376066019587147e-05,0.3322809934616089,0.48097705841064453,146.14280700683594,160.37643698336922,13.979000811213632,3.8662649184685504e-05
5,11824,12701,11824-12701,211567,0.0570328748289802,2.5316271966844003e-05,0.3941173851490021,0.7089425325393677,104.9497299194336,183.2640361716123,7.67890289776719,4.585764006800459e-05
6,11757,12701,11757-12701,1095337,0.0389700499520749,2.564632846565043e-05,0.22686167061328888,0.8023555874824524,45.64091110229492,94.9256083589118,18.12270126187419,2.6396553991845033e-05
7,9031,12702,9031-12702,1205063,0.0403160857304091,3.021500622088505e-05,0.2188577502965927,0.9376584887504578,28.06189727783203,24.364068570208374,13.40843002785,2.546525557455426e-05
8,7981,12704,7981-12704,101204,0.0236797124276408,2.4383753498860976e-05,0.6746231317520142,0.21215707063674927,155.55569458007812,127.75123724670208,9.906489618974625,7.8495965727921e-05


# Temporarily set bad MaNGA galaxies to have -999 values

These were determined from VI of the MaNGA fits and maps:
* Galaxies with no data
    * failed T-type cut: 9862-12704 (T-type=-0.46)
    * failed smoothness cut:  8951-3704, 11835-6103, 11024-6102
      
* Too much data masked: 8999-6103, 9032-12703
* Merger: 11865-6102
* Bad fits: 8144-3703, 8335-12704, 8977-12704, 9498-12703, 10224-9101, 10497-6103, 11828-12705, 11865-1902, 11949-12702, 11978-12701, 12506-12701, 12512-3701, 12769-12705 
* Successfully refit: 12079-9101 (note mngtarg1 = 0, mngtarg3 > 0), 11955-3703 (note mngtarg1 = 0, mngtarg3 > 0),  '11835-12703', '11982-3702'

In [21]:
bad_gals = ['8999-6103', '9032-12703', '11865-6102', '8144-3703', 
            '8335-12704', '8977-12704', '9498-12703', '10224-9101', 
            '10497-6103', '11828-12705', '11865-1902', '11949-12702', 
            '11978-12701', '12506-12701', '12512-3701', '12769-12705']

for gal_ID in bad_gals:
    idx = np.where(MaNGA_fits['plateifu'] == gal_ID)[0]

    MaNGA_fits['v_max'][idx] = -999

# Add the MaNGA best-fit values to the table

In [50]:
SGA_MaNGA_dr1['Vmax_map'] = np.nan
SGA_MaNGA_dr1['Vmax_err_map'] = np.nan
SGA_MaNGA_dr1['Rturn_map'] = np.nan
SGA_MaNGA_dr1['alpha_map'] = np.nan

SGA_MaNGA_dr1['ba_map'] = np.nan
SGA_MaNGA_dr1['ba_err_map'] = np.nan
SGA_MaNGA_dr1['ba_NSA'] = np.nan

SGA_MaNGA_dr1['phi_map'] = np.nan
SGA_MaNGA_dr1['phi_err_map'] = np.nan
SGA_MaNGA_dr1['phi_NSA'] = np.nan

SGA_MaNGA_dr1['x0_map'] = np.nan
SGA_MaNGA_dr1['x0_err_map'] = np.nan
SGA_MaNGA_dr1['y0_map'] = np.nan
SGA_MaNGA_dr1['y0_err_map'] = np.nan

SGA_MaNGA_dr1['chi2_map'] = np.nan

SGA_MaNGA_dr1['Z_NSA'] = np.nan

for i in range(len(SGA_MaNGA_dr1)):
    
    gal_id = SGA_MaNGA_dr1['plateifu'][i]
    
    # Find galaxy row in MaNGA fits table
    i_fit = MaNGA_fits['plateifu'] == SGA_MaNGA_dr1['plateifu'][i]
    
    # Copy best-fit parameter values from fit table to galaxy table
    if (np.sum(i_fit) > 0):
        
        SGA_MaNGA_dr1['Vmax_map'][i] = MaNGA_fits['v_max'][i_fit]
        SGA_MaNGA_dr1['Vmax_err_map'][i] = MaNGA_fits['v_max_err'][i_fit]
        SGA_MaNGA_dr1['Rturn_map'][i] = MaNGA_fits['r_turn'][i_fit]
        SGA_MaNGA_dr1['alpha_map'][i] = MaNGA_fits['alpha'][i_fit]
        
        SGA_MaNGA_dr1['ba_map'][i] = MaNGA_fits['ba'][i_fit]
        SGA_MaNGA_dr1['ba_err_map'][i] = MaNGA_fits['ba_err'][i_fit]
        SGA_MaNGA_dr1['ba_NSA'][i] = MaNGA_fits['nsa_elpetro_ba'][i_fit]
        
        SGA_MaNGA_dr1['phi_map'][i] = MaNGA_fits['phi'][i_fit]
        SGA_MaNGA_dr1['phi_err_map'][i] = MaNGA_fits['phi_err'][i_fit]
        SGA_MaNGA_dr1['phi_NSA'][i] = MaNGA_fits['nsa_elpetro_phi'][i_fit]

        SGA_MaNGA_dr1['x0_map'][i] = MaNGA_fits['x0'][i_fit]
        SGA_MaNGA_dr1['x0_err_map'][i] = MaNGA_fits['x0_err'][i_fit]
        SGA_MaNGA_dr1['y0_map'][i] = MaNGA_fits['y0'][i_fit]
        SGA_MaNGA_dr1['y0_err_map'][i] = MaNGA_fits['y0_err'][i_fit]

        SGA_MaNGA_dr1['chi2_map'][i] = MaNGA_fits['chi2'][i_fit]
        
        SGA_MaNGA_dr1['Z_NSA'][i] = MaNGA_fits['nsa_z'][i_fit]
        

# Flip all -99 values to NaN
for col_name in SGA_MaNGA_dr1.colnames:
    
    bad_values = SGA_MaNGA_dr1[col_name] == -999
    
    if np.any(bad_values):
        SGA_MaNGA_dr1[col_name][bad_values] = np.nan

/Users/kdouglass/miniforge3/envs/desi/lib/python3.10/site-packages/astropy/table/column.py:1376: UserWarning: Warning: converting a masked element to nan.
  self.data[index] = value


In [51]:
good_V = np.isfinite(SGA_MaNGA_dr1['Vmax_map']) & np.isfinite(SGA_MaNGA_dr1['V_0p4R26'])

print(np.sum(good_V))

257


In [52]:
# Also remove those galaxies from the final sample that have V(R26) < 0.9Vmax, and have Vmax > 1000 km/s

# 1 - Convert R26 to kpc for each galaxy
Planck18_h = FlatLambdaCDM(H0=100, Om0=0.3151)
dist_to_galaxy = Distance(z=SGA_MaNGA_dr1['Z_DESI'], cosmology=Planck18_h)
R26_kpc = dist_to_galaxy.to('kpc')*np.tan((SGA_MaNGA_dr1['R26']*u.arcmin).to(u.rad))

# 2 - Compute V(R26)
SGA_MaNGA_dr1['Vfit_R26'] = rot_fit_BB(R26_kpc.data, 
                                       [SGA_MaNGA_dr1['Vmax_map'], 
                                        SGA_MaNGA_dr1['Rturn_map'], 
                                        SGA_MaNGA_dr1['alpha_map']])

# 3 - Filter out those with V(R26) < 0.9Vmax
goodVmax = SGA_MaNGA_dr1['Vfit_R26'] >= 0.9*SGA_MaNGA_dr1['Vmax_map']

# 4 - Filter out those with Vmax > 1000 km/s
lowVmax = SGA_MaNGA_dr1['Vmax_map'] < 1000.

final_sample = good_V & goodVmax & lowVmax

In [53]:
SGA_MaNGA_dr1_final = SGA_MaNGA_dr1[final_sample]
SGA_MaNGA_dr1_final

plate,ifudsgn,plateifu,SGA_ID,Z_DESI,ZERR_DESI,R26,BA,PA,V_0p4R26,V_0p4R26_ERR,SKY_FIBER_DIST,Vmax_map,Vmax_err_map,Rturn_map,alpha_map,ba_map,ba_err_map,ba_NSA,phi_map,phi_err_map,phi_NSA,Z_NSA,Vfit_R26,chi2_map,x0_map,x0_err_map,y0_map,y0_err_map
,,,,,,,,,,,rad,,,,,,,,,,,,,,,,,
int64,int64,str11,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
9045,3702,9045-3702,29152,0.033320236763565,2.479367968152881e-05,0.31793898344039917,0.6588062047958374,168.80343627929688,136.65487767299953,11.458479382493653,3.699388054319129e-05,210.50452801297985,1.8353976854896468,0.7066919287420622,0.9111401871299377,0.8234098343647784,0.001601497928492459,0.70429,170.71749392618315,0.02929622324244995,168.862,0.0333513,190.7436504644614,46.16357347070347,21.882076864143773,0.005466208942145591,22.188064369156837,0.007755939838456064
8615,1901,8615-1901,1265162,0.0197616666129208,2.4068447363858412e-05,0.2447134554386139,0.9234866499900818,75.16405487060547,142.94633456308551,29.494479148008217,2.8473703475574915e-05,220.16802918955494,0.938473172075278,0.9732954258408208,2.5112860326014714,0.9557675585663513,0.00040024946764247393,0.864664,116.18630266418596,0.03337353804379676,29.9028,0.0197432,218.07771302058381,254.08051342996677,15.766596851398912,0.01848932560602464,14.491757466046138,0.022291160399820775
7958,3702,7958-3702,1279893,0.0370122558474312,2.820956877908207e-05,0.4380972981452942,0.6277525424957275,142.59364318847656,219.66869829372382,14.440870762250226,5.0974935311510876e-05,242.9211676900525,0.1875765246066455,1.365763143678639,3.082085057706107,0.678949712986619,0.0005240797629047349,0.719594,321.0557249418591,0.015508704418586036,138.715,0.0370471,242.86733688210546,32.086971113181455,22.231848341492075,0.004116164357321055,22.113666689527047,0.004096986301522966
11834,6104,11834-6104,1002687,0.0440140222262614,2.6774924836553197e-05,0.2553322911262512,0.4506562352180481,161.87460327148438,152.18986377456818,6.239301721442626,2.9709261112091895e-05,232.29567853040538,3.5372042573275575,1.6157067061530481,2.2182435957133957,0.7980107504115111,0.005971267624381377,0.7131611,162.8144980146715,0.1086854487008954,160.61732,0.044045065,230.53071367484205,4.049425591865072,24.486365545597476,0.015233610676827455,36.484026774689966,0.02462972571986479
12081,12702,12081-12702,929612,0.0418290423138136,2.9376066019587147e-05,0.3322809934616089,0.48097705841064453,146.14280700683594,160.37643698336922,13.979000811213632,3.8662649184685504e-05,195.42432287880288,1.3346281500087767,1.3198023994720334,1.1216729306540982,0.6541854019632092,0.003249980121659414,0.5533,324.31771976680415,0.10032848698407756,145.41603,0.041768476,182.42008819109282,2.95303632544946,36.257802459299405,0.024843183463947694,36.251797542590616,0.026449329132018303
11824,12701,11824-12701,211567,0.0570328748289802,2.5316271966844003e-05,0.3941173851490021,0.7089425325393677,104.9497299194336,183.2640361716123,7.67890289776719,4.585764006800459e-05,213.21030009683315,0.37995901749603017,3.2421984975912386,4.107502810453571,0.7709798023865316,0.000980767615119872,0.79347116,277.7050698514268,0.022168037157852916,112.61641,0.05706307,213.18333753615693,13.838433863696032,37.21675625452923,0.00973518309397235,36.962858957869834,0.005981815196063073
11757,12701,11757-12701,1095337,0.0389700499520749,2.564632846565043e-05,0.22686167061328888,0.8023555874824524,45.64091110229492,94.9256083589118,18.12270126187419,2.6396553991845033e-05,139.8589447091876,3.48141965647591,1.9702654263831174,3.565549293643847,0.9012611974513942,0.005217860723617648,0.87782425,44.92909939483331,0.11326059485829489,45.205704,0.038932998,139.58715853046652,3.6016611258181985,35.534803321823844,0.04576396631947133,35.621518361285695,0.04558715486014384
7981,12704,7981-12704,101204,0.0236797124276408,2.4383753498860976e-05,0.674623131752

## Find repeat galaxies in the MaNGA observations and keep the best $\chi^2$ fit

In [54]:
unique_manga, unique_manga_counts = np.unique(SGA_MaNGA_dr1_final['SGA_ID'], return_counts=True)

repeat_manga_sga_ids = unique_manga[unique_manga_counts > 1]

bad_repeats = np.ones(len(SGA_MaNGA_dr1_final), dtype=bool)

for sga_id in repeat_manga_sga_ids:

    # Find all observations with this SGA ID
    obs = SGA_MaNGA_dr1_final['SGA_ID'] == sga_id
    plateifus = SGA_MaNGA_dr1_final['plateifu'][obs]

    # Identify which of these has the minimum chi2
    good_idx = np.argmin(SGA_MaNGA_dr1_final['chi2_map'][obs])

    # Create boolean to identify all those which need to be removed
    cut = np.ones(len(plateifus), dtype=bool)
    cut[good_idx] = 0
    
    # Set all other fits of this galaxy to 0 (to be removed)
    for plateifu in plateifus[cut]:
        bad_repeats[SGA_MaNGA_dr1_final['plateifu'] == plateifu] = 0

final_MaNGA_sample = SGA_MaNGA_dr1_final[bad_repeats]

# Model DESI observations

For each galaxy that is in our Iron sample AND has a MaNGA velocity map model, 
1. Identify which observations are of this galaxy (both center and $0.4R_{26}$
2. Filter DESI observations to keep only those that went into the calculation of $V(0.4R_{26})$ (both centers and those at $0.4R_{26}$)
3. For each good DESI observation, compute the expected velocity from the MaNGA velocity map model
4. Combine the predicted velocities in a similar manner to how we compute $V(0.4R_{26})$ from the DESI observations

We will do all these steps on a galaxy-by-galaxy basis.  (I think it will be easier to keep track of everything.)

In [55]:
# Function to identify which DESI observations belong to this galaxy
def identify_DESI_obs(DESI_obs, sga_id):
    '''
    For the given sga_id, find all DESI observations taken of this galaxy.
    '''
    gal_idx = DESI_obs['SGA_ID'] == sga_id

    return DESI_obs[gal_idx]

In [56]:
def check_sides(galaxies):
    '''
    Check that the rotational velocities have the same sign on each side of the 
    galaxy.
    '''
    # Find which observations are on the same, opposite side
    same_side = (galaxies['TARGET_RA'] == galaxies['TARGET_RA'][0]) & (galaxies['TARGET_DEC'] == galaxies['TARGET_DEC'][0])

    # Check that all velocities on the same side as the first in the table have 
    # the same sign
    same1 = galaxies['V_ROT'][same_side] > 0
    check_same = np.all(same1 == same1[0])

    # Check if there are observations on the opposite side of the galaxy from 
    # the first one in the table
    if np.sum(same_side) < len(galaxies):

        # Check that all the velocities on this size have the same sign
        same2 = galaxies['V_ROT'][~same_side] > 0
        check_same2 = np.all(same2 == same2[0])

        # Check that the signs are opposite each other on the two sides
        check_opposite = check_same & check_same2 & (same1[0] != same2[0])

        # Combine all the checks into one boolean value
        check_same = check_same & check_same2 & check_opposite

    return check_same

In [57]:
# Function to filter the DESI observations to keep only those that went into the 
# calculation of V(0.4R26)
def clean_DESI_obs(DESI_obs, gal_coords, D26, ba):
    '''
    Return two sets of observations - centers and those at 0.4R26 - that survive 
    all quality cuts.

    Note that gal_coords should be a SkyCoord object, and D26 needs units.
    '''

    ############################################################################
    # 1. Calculate the on-sky separation between each observation and the 
    #    galaxy's center
    #---------------------------------------------------------------------------
    target_coords = SkyCoord(ra=DESI_obs['RA'], 
                             dec=DESI_obs['DEC'], 
                             unit=u.degree)
    
    DESI_obs['SKY_FIBER_DIST'] = target_coords.separation(gal_coords)
    
    DESI_obs['SKY_FIBER_DIST_R26'] = 2*DESI_obs['SKY_FIBER_DIST'].to('arcmin')/D26
    ############################################################################


    ############################################################################
    # 2. Separate into center and off-center observations
    #---------------------------------------------------------------------------
    # Centers
    centers_boolean = DESI_obs['SKY_FIBER_DIST_R26'] < 0.1
    DESI_centers = DESI_obs[centers_boolean]

    # 0.4R26
    axis_boolean = (DESI_obs['SKY_FIBER_DIST_R26'] > 0.38) & (DESI_obs['SKY_FIBER_DIST_R26'] < 0.42)
    DESI_axis = DESI_obs[axis_boolean]
    ############################################################################


    ############################################################################
    # 3. Clean the center observations
    #---------------------------------------------------------------------------
    good_centers = DESI_centers[(DESI_centers['DELTACHI2'] > 25) & (DESI_centers['ZWARN'] == 0)]
    ############################################################################


    ############################################################################
    # 4. Calculate the rotational velocities of the off-center observations
    #---------------------------------------------------------------------------
    # Calculate the average center redshift
    z_center = np.average(good_centers['Z'], 
                          weights=good_centers['ZERR_MOD']**-2)

    # Calculate the uncertainty in the center redshift
    z_random = np.zeros((len(good_centers), 10000))

    for i in range(len(good_centers)):
        z_random[i] = rng.normal(loc=good_centers['Z'][i], 
                                 scale=good_centers['ZERR_MOD'][i], 
                                 size=10000)

    z_center_random = np.average(z_random, 
                                 weights=np.ones(10000)*(good_centers['ZERR_MOD'][:,None]**-2), 
                                 axis=0)
    z_err_center = np.std(z_center_random)

    # Calculate rotational velocity for each observation
    z_rot = (1 + DESI_axis['Z'])/(1 + z_center) - 1
    DESI_axis['V_ROT'] = c*z_rot

    # Calculate uncertainty in rotational velocity for each observation
    z_axis_random = np.zeros((len(good_centers), 10000))

    for i in range(len(good_centers)):
        z_axis_random[i] = rng.normal(loc=DESI_axis['Z'][i], 
                                      scale=DESI_axis['ZERR_MOD'][i], 
                                      size=10000)

    z_rot_random = (1 + z_axis_random)/(1 + z_center_random) - 1

    DESI_axis['V_ROT_ERR'] = np.std(np.abs(c*z_rot_random), axis=1)
    ############################################################################


    ############################################################################
    # 5. Correct rotational velocities for inclination angle
    #---------------------------------------------------------------------------
    cosi2 = (ba**2 - q0**2)/(1 - q0**2)

    if cosi2 < 0:
        cosi2 = 0

    DESI_axis['V_ROT'] /= np.sin(np.arccos(np.sqrt(cosi2)))
    DESI_axis['V_ROT_ERR'] /= np.sin(np.arccos(np.sqrt(cosi2)))
    ############################################################################


    ############################################################################
    # 6. Keep only those with 10 < Vrot < 1000 km/s
    #---------------------------------------------------------------------------
    Vgood = (np.abs(DESI_axis['V_ROT']) < 1000) & (np.abs(DESI_axis['V_ROT']) > 10)

    good_axis = DESI_axis[Vgood]
    ############################################################################


    ############################################################################
    # 7. Keep only those with dV/Vmin <= 5
    #---------------------------------------------------------------------------
    good_deltaV = np.ones(len(good_axis), dtype=bool)
    
    if len(good_axis) > 1:
    
        deltachi2_idx = good_axis['DELTACHI2'] >= 25

        Vmin = np.min(np.abs(good_axis['V_ROT']))

        v_norm_min = np.abs(good_axis['V_ROT'])/Vmin

        diff_matrix = np.abs(good_axis['V_ROT']).reshape(len(good_axis), 1) - np.abs(good_axis['V_ROT']).reshape(1, len(good_axis))

        diff_matrix_norm = diff_matrix/Vmin

        if np.any(np.abs(diff_matrix_norm) > 5):

            # Remove all observations with DELTACHI2 < 25
            # Note: This also typically removes observations with ZWARN != 0
            good_deltaV[~deltachi2_idx] = False

            # Check to make sure that, if there are still multiple observations, 
            # they all satisfy our relative velocity criteria
            if np.sum(deltachi2_idx) > 1:

                Vmin = np.min(np.abs(good_axis['V_ROT'][deltachi2_idx]))

                diff_matrix = np.abs(good_axis['V_ROT'][deltachi2_idx]).reshape(np.sum(deltachi2_idx), 1) - np.abs(good_axis['V_ROT']).reshape(1, np.sum(deltachi2_idx))

                diff_matrix_norm = diff_matrix/Vmin

                if np.any(np.abs(diff_matrix_norm) > 5):

                    # Set all these so that we do not look at this galaxy
                    # Note there shouldn't be any that make it in here, because 
                    # all the objects in this file are ones with good final 
                    # velocities
                    good_deltaV[deltachi2_idx] = False

    good_deltaV_axis = good_axis[good_deltaV]
    ############################################################################


    ############################################################################
    # 8. Keep only those with the same velocity sign on the same side (and 
    #    opposite sign on the opposite side)
    #---------------------------------------------------------------------------
    same_side_good = np.ones(len(good_deltaV_axis), dtype=bool)

    if len(good_deltaV_axis) > 1:

        deltachi2_idx = good_deltaV_axis['DELTACHI2'] >= 25

        check_same = check_sides(good_deltaV_axis)

        if ~check_same:
            # At least one of the observations is the wrong sign

            # First, remove all with DELTACHI2 < 25
            same_side_good[~deltachi2_idx] = False

            if np.sum(good_obs_idx) > 1:

                # Recheck those that are left
                check_same_again = check_sides(good_deltaV_axis[deltachi2_idx])

                # If this still fails, remove all observations
                # Note that this should never happen here, because all these 
                # galaxies have good final velocities
                if ~check_same_again:
                    same_side_good[deltachi2_idx] = False

    good_deltaV_axis_same = good_deltaV_axis[same_side_good]
    ############################################################################
    
    return good_centers, good_deltaV_axis_same

In [ ]:
def MaNGA_vel(DESI_obs, MaNGA_fit):
    '''
    Calculate the predicted rotational velocity from the best-fit MaNGA velocity 
    map model.
    '''

    ############################################################################
    # 1. Calculate the deprojected radius between the target and the center of 
    # the velocity map, and the position angle os the observation
    #---------------------------------------------------------------------------
    # Retrieve the WCS of the MaNGA data cube
    
    wcs_map = 
    
    # Convert the (ra,dec) positions of the DESI observations to pixel 
    # coordinates
    DESI_coords = SkyCoord(ra=DESI_obs['RA'], dec=DESI_obs['DEC'], unit=u.degree)
    xy_obs = skycoord_to_pixel(DESI_coords, wcs_map)

    # Calculate deprojected radius
    r_deproj = np.zeros(len(DESI_coords))
    theta = np.zeros(len(DESI_coords))

    cosi2_SGA = (MaNGA_fit['BA']**2 - q0**2)/(1 - q0**2)

    for i in range(len(DESI_coords)):
        r_deproj[i], theta[i] = deproject_spaxel(xy_obs[i], 
                                                 (MaNGA_fit['x0'], MaNGA_fit['y0']), 
                                                 (MaNGA_fit['phi_map']*u.degree).to(u.radian), 
                                                 np.sin(np.arccos(np.sqrt(cosi2_SGA))))

    # Convert radius units from pixels to kpc
    dist_to_galaxy = cosmo.comoving_distance(MaNGA_fit['Z_DESI'])
    r_deproj_kpc = r_deproj*dist_to_galaxy.to('kpc')*np.tan(MANGA_SPAXEL_SIZE)
    ############################################################################


    
    ############################################################################
    # 2. Calculate the tangential velocity at this deprojected radius
    #---------------------------------------------------------------------------
    cosi2_MaNGA = (MaNGA_fit['ba_map']**2 - q0**2)/(1 - q0**2)
    
    Vmax_SGA = MaNGA_fit['Vmax_map']*np.sin(np.arccos(np.sqrt(cosi2_MaNGA)))/np.sin(np.arccos(np.sqrt(cosi2_SGA)))

    V_MaNGA = rot_fit_BB(r_deproj_kpc, 
                         [Vmax_SGA, 
                          MaNGA_fit['Rturn_map'], 
                          MaNGA_fit['alpha_map']])
    ############################################################################


    ############################################################################
    # 3. Adjust the tangentail velocity to the observed velocity
    #---------------------------------------------------------------------------
    DESI_obs['V_MaNGA'] = V_MaNGA*np.sin(np.arccos(np.sqrt(cosi2_SGA)))*np.cos(pa_obs - MaNGA_fit['PA']*u.degree)
    ############################################################################

In [ ]:
final_MaNGA_sample['V_0p4R26_MaNGA'] = np.nan

SGA_coords = SkyCoord(ra=final_MaNGA_sample['RA'], 
                      dec=final_MaNGA_sample['DEC'], 
                      unit=u.degree)

for i in range(len(final_MaNGA_sample)):

    # Find all DESI observations of this galaxy
    DESI_targets = identify_DESI_obs(tiron_goodVI, 
                                     final_MaNGA_sample['SGA_ID'][i])

    # Separate DESI observations into centers and off-center observations, and 
    # remove all which do not pass quality cuts
    good_DESI_centers, good_DESI_0p4R26 = clean_DESI_obs(DESI_targets, 
                                                         SGA_coords[i], 
                                                         final_MaNGA_sample['D26'][i]*u.arcmin, 
                                                         final_MaNGA_sample['BA'][i])

    # For each good DESI observation, calculate the predicted velocity from the 
    # MaNGA model velocity map
    MaNGA_V_center = MaNGA_vel(good_DESI_centers, 
                               final_MaNGA_sample[i])

    # Combine the predicted velocities to compute a total rotational velocity
    